In [9]:
import jax
from jax import flatten_util, vmap 
import jax.numpy as jnp
import jax.random as jr
import os
import matplotlib.pyplot as plt
from physical_diffusion_fns.helper_fns import sample_gaussian_mixture, normalize_samples, sample_forward_process, get_best_params, smooth_parameters, interpolate_parameters
from physical_diffusion_fns.learning_fns import setup_MLE_loss_per_batch, setup_MLE_gradient_per_batch, CD1_gradient, setup_score_matching_loss_per_batch, run_optimization
from physical_diffusion_fns.plotting_fns import plot_energy_and_distributions, plot_parameter_evolution, plot_forward_marginals, visualize_connectivity, plot_parameter_as_fn_of_time
from physical_diffusion_fns.network_fns import setup_duffing_network_with_external_force_energy_fn, setup_overdamped_SDE, solve_SDE, create_2d_square_grid_connectivity


jax.config.update("jax_enable_x64", True)

### Load MNIST and pre-process

In [10]:
training_method = "SM"
rescaling = False
sigma_images = 0.05
adding_noise = True
additional_rescaling = 1.
# Forward process parameters
n_time_steps = 30
t_forward = 1.0
sigma_forward = 1.
forward_time_pts = jnp.exp(jnp.linspace(jnp.log(1e-9), jnp.log(t_forward), n_time_steps))
forward_time_pts = forward_time_pts.at[0].set(0.)
print('forward_time_pts', forward_time_pts)

# Optimization parameters
learning_rate = 0.1
n_epochs = 100000//2
batch_size = 128
window_size=1000
tolerance=1e-8
patience=50
comment = f""
key_seed = 0
N_osc = 64
connectivity = create_2d_square_grid_connectivity(grid_size=8)

forward_time_pts [0.00000000e+00 2.04335972e-09 4.17531894e-09 8.53167852e-09
 1.74332882e-08 3.56224789e-08 7.27895384e-08 1.48735211e-07
 3.03919538e-07 6.21016942e-07 1.26896100e-06 2.59294380e-06
 5.29831691e-06 1.08263673e-05 2.21221629e-05 4.52035366e-05
 9.23670857e-05 1.88739182e-04 3.85662042e-04 7.88046282e-04
 1.61026203e-03 3.29034456e-03 6.72335754e-03 1.37382380e-02
 2.80721620e-02 5.73615251e-02 1.17210230e-01 2.39502662e-01
 4.89390092e-01 1.00000000e+00]


In [ ]:
import numpy as np
# Load the saved .npy file
data_np = np.load("data/MNIST/MNIST_0_1_8x8pix/mnist_0_1_8x8pix.npy")
key =  jr.PRNGKey(key_seed)
# Optionally convert to a JAX array
images_flat_raw = jnp.array(data_np)
n_samples = images_flat_raw.shape[0]
print("Data shape:", images_flat_raw.shape)

In [4]:
if rescaling:
    samples_target, mean_MNIST, std_MNIST = normalize_samples(images_flat_raw)
    samples_target = samples_target*additional_rescaling
elif adding_noise:    
    key_images =  jr.PRNGKey(100)  # Seed for reproducibility
    gaussian_noise = jr.normal(key_images, images_flat_raw.shape) * sigma_images 
    images_flat_raw_noised = images_flat_raw + gaussian_noise
    samples_target, mean_MNIST, std_MNIST = normalize_samples(images_flat_raw_noised)
else:
    samples_target, mean_MNIST, std_MNIST = normalize_samples(images_flat_raw)

In [ ]:
images_true = samples_target.reshape(-1, 8, 8)*std_MNIST + mean_MNIST
# Plot a few examples
num_examples = 10  # number of examples to display
fig, axes = plt.subplots(1, num_examples, figsize=(15, 2))

for i in range(num_examples):
    # Remove channel dimension if it exists (i.e., converting 8x8x1 to 8x8)
    img = images_true[i]
    if img.shape[-1] == 1:
        img = img.squeeze(-1)
    
    axes[i].imshow(np.array(img), cmap='gray')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

### Network

In [ ]:
num_connections = connectivity.shape[0]
visualize_connectivity(connectivity, grid_size_x=8, grid_size_y=8)
k_lin_0 = -1.*jnp.ones(N_osc)
k_duff_0 = jnp.ones(N_osc)
c_lin_0 = jnp.zeros(num_connections)
c_optomech_0 = jnp.zeros(num_connections)
biases_0 = jnp.zeros(N_osc)
params_initial = (k_lin_0, k_duff_0, c_lin_0, c_optomech_0, biases_0)
params_names = ['k_lin', 'k_duff', 'c_lin', 'c_optomech', 'biases']
params_flattened_initial, unflatten = flatten_util.ravel_pytree(params_initial)

### Bringe energy into correct form

In [8]:
energy_fn = setup_duffing_network_with_external_force_energy_fn(connectivity, unflatten)

### Plot initial energy landscape

In [9]:
# Setup loss and gradient functions based on selected method
if training_method == "SM":
    loss_fn_per_batch = setup_score_matching_loss_per_batch(energy_fn)
    gradient_fn_per_batch = None
    maximize = False
elif training_method == "CD1":
    loss_fn_per_batch = setup_score_matching_loss_per_batch(energy_fn)
    gradient_fn_per_batch = lambda flattened_args, batch: -CD1_gradient(energy_fn, batch, flattened_args, dt=0.001, D=1, key=jnp.random.PRNGKey(1535), num_noise_samples=1000)
    maximize = False
elif training_method == "MLE":
    loss_fn_per_batch = setup_MLE_loss_per_batch(energy_fn)
    gradient_fn_per_batch = setup_MLE_gradient_per_batch(energy_fn)
    maximize = True
else:
    raise ValueError(f"Unknown training method: {training_method}. Choose from 'SM', 'CD1', or 'MLE'.")

#### Optimization parameters and filenames

In [ ]:
data_folder = ("normalized_data_"
    f"{f'adding_noise_{sigma_images}' if adding_noise else 'NO_added_noise_'}"
    f"{f'rescaling_{additional_rescaling}' if rescaling else 'NO_additional_rescaling_'}"
).strip('_')

comment = f""

optimization_folder = (f"{training_method}_"
           f"t_forward_{t_forward}_"
           f"n_timesteps_{n_time_steps}_"
           f"sigma_forward_{sigma_forward}_"
           f"lr_{learning_rate}_"
           f"epochs_{n_epochs}_"
           f"batch_{batch_size}_"
           f"window_{window_size}_"
           f"tol_{tolerance}_"
           f"patience_{patience}_"
           f"{comment}")

# Setup output directories
base_dir = "out/problems"
problem_type_folder = "MNIST"
problem_folder = f"only_0_and_1_resol_8x8/{data_folder}"  # Replace with your actual parameters

output_dir = os.path.join(base_dir, problem_type_folder, problem_folder,optimization_folder)

# Create directories if they don't exist
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

# Load existing parameters
params_history_all_t = jnp.load(f"{output_dir}/params_history.npy")

### Print final distribution of the forward

In [ ]:
# key, subkey = jr.split(key)
# samples_t = sample_forward_process(t_forward, n_samples, D=1, sigma_final=sigma_forward, samples0=samples_target, key=subkey)
# plot_forward_marginals(samples_t, t_forward, sigma_forward, beta=1.0, path=output_dir, save_fig=True, fontsize=16)

### Smoothen and interpolate parameter evolution

In [ ]:
use_smoothed_params = False
if use_smoothed_params:
    # First smooth the parameters
    smoothed_params = smooth_parameters(params_history_all_t, window_lengths = [10] * params_history_all_t.shape[1], poly_orders = [3] * params_history_all_t.shape[1])
    # Get interpolator function
    params_interpolator = interpolate_parameters(smoothed_params, forward_time_pts)
else:
    # Get interpolator function
    params_interpolator = interpolate_parameters(params_history_all_t, forward_time_pts)

# Then interpolate the smoothed parameters
time_eval = jnp.linspace(forward_time_pts[0],forward_time_pts[-1],200)
# interpolated_params = interpolate_parameters(smoothed_params, forward_time_pts, time_dense)
interpolated_params = params_interpolator(time_eval)


plot_parameter_as_fn_of_time(params_names, forward_time_pts,forward_time_pts, params_history_all_t, params_interpolator, unflatten, N_osc) 

### Analize params

In [ ]:
# Assuming params_interpolator is defined and returns the correct values
k_lin_t_0 = params_interpolator(0.0)[0:N_osc]
k_duff_t_0 = params_interpolator(0.0)[N_osc:2*N_osc]

# Reshape to 8x8
k_lin_t_0_reshaped = k_lin_t_0.reshape((8, 8))
k_duff_t_0_reshaped = k_duff_t_0.reshape((8, 8))

# Plotting
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# Plot k_lin_t_0
cax1 = ax[0].imshow(k_lin_t_0_reshaped, cmap='viridis', aspect='equal')
ax[0].set_title('k_lin_t_0')
fig.colorbar(cax1, ax=ax[0])

# Plot k_duff_t_0
cax2 = ax[1].imshow(k_duff_t_0_reshaped, cmap='viridis', aspect='equal')
ax[1].set_title('k_duff_t_0')
fig.colorbar(cax2, ax=ax[1])

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.colors import LogNorm# Assuming params_interpolator is defined and returns the correct values
k_lin_t_0 = params_interpolator(0.0)[0:N_osc]
k_duff_t_0 = params_interpolator(0.0)[N_osc:2*N_osc]

# Reshape to 8x8
k_lin_t_0_reshaped = k_lin_t_0.reshape((8, 8))
k_duff_t_0_reshaped = k_duff_t_0.reshape((8, 8))

# Plotting
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# Plot k_lin_t_0 with logarithmic scale
cax1 = ax[0].imshow(jnp.abs(k_lin_t_0_reshaped), cmap='viridis', norm=LogNorm(), aspect='equal')
ax[0].set_title('abs k_lin_t_0 (Log Scale)')
fig.colorbar(cax1, ax=ax[0])

# Plot k_duff_t_0 with logarithmic scale
cax2 = ax[1].imshow(jnp.abs(k_duff_t_0_reshaped), cmap='viridis', norm=LogNorm(), aspect='equal')
ax[1].set_title('abs k_duff_t_0 (Log Scale)')
fig.colorbar(cax2, ax=ax[1])

plt.tight_layout()
plt.show()

### Run reverse SDE:

In [19]:
forward_params = jnp.zeros_like(params_flattened_initial)
forward_params = forward_params.at[0:N_osc].set(1.)
params_interpolator_reverse_plus_linear = lambda t: 2*params_interpolator(t_forward - t) - 1/sigma_forward**2 * forward_params
# Setup SDE functions
drift_fn, diffusion_fn = setup_overdamped_SDE(energy_fn, params_interpolator_reverse_plus_linear, N_osc, time_dependent_parms=True)

In [ ]:
t0 = 0.0
t1 = t_forward
ts = jnp.linspace(t0, t1, 100)
dt0 = 0.00000001

# Generate multiple initial states
n_trajectories = 10
key, subkey = jr.split(key)
initial_states = sample_forward_process(t_forward, n_trajectories, sigma_final=sigma_forward, D=1, samples0=samples_target, key=subkey)

# Generate a key for each initial condition
key, subkey = jr.split(key)
keys_brownian = jr.split(subkey, n_trajectories)

# Vectorize solve_SDE across both initial states and keys
vectorized_solve_SDE = vmap(
    lambda init_state, key_b: solve_SDE(
        drift_fn, 
        diffusion_fn, 
        init_state,
        key_b, 
        t0, 
        t1, 
        ts.shape[0], 
        dt0,
        rtol=1e-6,
        atol=1e-9
    ),
    in_axes=(0, 0)
)


# Run SDE for all initial states at once
solutions = vectorized_solve_SDE(initial_states, keys_brownian)
all_trajectories = solutions.ys  # Shape: (n_trajectories, n_timesteps, N_osc)
final_states = all_trajectories[:, -1, :]  # Shape: (n_trajectories, N_osc)
final_states_rescaled = final_states  * std_MNIST + mean_MNIST
images_generated = final_states_rescaled.reshape(-1, 8, 8)

### Plot generated images vs true images

In [ ]:
# Plot a few examples
num_examples = 10  # number of examples to display
fig, axes = plt.subplots(2, num_examples, figsize=(15, 4))  # Create a 2-row grid

# Plot true images
for i in range(num_examples):
    img = images_true[i]
    if img.shape[-1] == 1:
        img = img.squeeze(-1)
    im = axes[0, i].imshow(np.array(img), cmap='gray')
    axes[0, i].axis('off')

# Add color bar for true images
fig.colorbar(im, ax=axes[0, :], orientation='horizontal', fraction=0.02, pad=0.04)

# Plot generated images
for i in range(num_examples):
    img_generated = images_generated[i]
    if img_generated.shape[-1] == 1:
        img_generated = img_generated.squeeze(-1)
    im = axes[1, i].imshow(np.array(img_generated), cmap='gray')
    axes[1, i].axis('off')

# Add color bar for generated images
fig.colorbar(im, ax=axes[1, :], orientation='horizontal', fraction=0.02, pad=0.04)

# Add titles
axes[0, 0].set_title("True images", loc='left', fontsize=12)
axes[1, 0].set_title("Generated images", loc='left', fontsize=12)

plt.tight_layout()
plt.show()


### Equillibirum Sampling

In [ ]:
# Get parameters from the first time step for equilibrium sampling
params_for_equillibrium_sampling = params_history_all_t[0,:]
# Setup SDE functions
drift_equilibrium_fn, diffusion_equilibrium_fn = setup_overdamped_SDE(energy_fn, params_for_equillibrium_sampling, N_osc, time_dependent_parms=False)

t0 = 0.0
t1 = 10
ts = jnp.linspace(t0, t1, 100)
dt0 = 0.00000001

# Generate multiple initial states
n_trajectories = 10  # or any other number of trajectories you need
n_dimensions = N_osc  # assuming 2D, adjust as necessary
key, subkey = jr.split(key)
initial_states = jr.normal(subkey, (n_trajectories, n_dimensions))

# Generate a key for each initial condition
key, subkey = jr.split(key)
keys_brownian = jr.split(subkey, n_trajectories)

# Vectorize solve_SDE across both initial states and keys
vectorized_solve_SDE = vmap(
    lambda init_state, key_b: solve_SDE(
        drift_equilibrium_fn, 
        diffusion_equilibrium_fn, 
        init_state,
        key_b, 
        t0, 
        t1, 
        ts.shape[0], 
        dt0,
        rtol=1e-4,
        atol=1e-7
    ),
    in_axes=(0, 0)
)


# Run SDE for all initial states at once
solutions_equilibrium = vectorized_solve_SDE(initial_states, keys_brownian)
all_trajectories_equilibrium = solutions_equilibrium.ys  # Shape: (n_trajectories, n_timesteps, 2)


In [ ]:
# Assuming `all_trajectories_equilibrium` is a JAX array with shape (n_trajectories, n_timesteps, n_dofs)
n_trajectories, n_timesteps, n_dofs = all_trajectories_equilibrium.shape

# Randomly select a subset of DOFs using JAX
key, subkey = jr.split(key)
n_dofs_to_plot = 5  # Number of DOFs to plot
random_dofs = jr.choice(subkey, jnp.arange(n_dofs), (n_dofs_to_plot,), replace=False)

# Convert to numpy for plotting
all_trajectories_equilibrium_np = jnp.array(all_trajectories_equilibrium)

# Plot the selected DOFs over time for each trajectory
plt.figure(figsize=(6, 6))
for dof in random_dofs:
    for traj in range(n_trajectories):
        plt.plot(all_trajectories_equilibrium_np[traj, :, dof])

plt.xlabel('Time Step')
plt.ylabel('DOF Value')
plt.title('Random Subset of DOFs Over Time')
plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))
plt.tight_layout()
plt.show()

In [ ]:
final_states_equilibrium = all_trajectories_equilibrium[:, -1, :]
all_trajectories_equilibrium_rescaled = all_trajectories_equilibrium  * std_MNIST + mean_MNIST
images_equilibrium_generated = all_trajectories_equilibrium_rescaled.reshape(-1, 8, 8)
# Plot a few examples
num_examples = 10  # number of examples to display
fig, axes = plt.subplots(2, num_examples, figsize=(15, 4))  # Create a 2-row grid

# Plot true images
for i in range(num_examples):
    img = images_true[i]
    if img.shape[-1] == 1:
        img = img.squeeze(-1)
    axes[0, i].imshow(np.array(img), cmap='gray')
    axes[0, i].axis('off')

# Plot generated images
for i in range(num_examples):
    img_generated = images_equilibrium_generated[i]
    if img_generated.shape[-1] == 1:
        img_generated = img_generated.squeeze(-1)
    axes[1, i].imshow(np.array(img_generated), cmap='gray')
    axes[1, i].axis('off')

# Add titles
axes[0, 0].set_title("True images", loc='left', fontsize=12)
axes[1, 0].set_title("Generated images", loc='left', fontsize=12)

plt.tight_layout()
plt.show()
